# 06. MMYOLO annotation JSON을 Label Studio project에 import

이미 annotation이 있는 MMYOLO/COCO-style JSON을 기존 Label Studio image task에 붙이는 notebook입니다. 사람이 눈으로 검수하거나, merge workflow에서 source prediction처럼 다루고 싶을 때 사용합니다.

안전 기본값은 `DRY_RUN=True`입니다. 이 상태에서는 task matching 수와 bbox 수만 계산하고 업로드하지 않습니다.

핵심 용어:
- `COCO_JSON`: import할 MMYOLO/COCO-style annotation JSON입니다.
- `image_match_mode="fullpath"`: JSON의 `file_name` 경로를 `MIRROR_ROOT` 기준 상대경로로 바꿔 task의 local-files 경로와 맞춥니다. 같은 파일명이 여러 폴더에 있을 때 안전합니다.
- `upload_mode="prediction"`: annotation이 있어도 Label Studio에는 prediction으로 넣습니다. merge/검수 workflow에서 다루기 편할 때가 많습니다.
- `META_TAG`, `IMPORT_ID`: 나중에 import 결과를 찾거나 삭제할 때 쓰는 식별자입니다.
- `BATCH_SIZE`: 너무 크게 잡으면 서버가 무거워질 수 있으므로 처음에는 작게 시작합니다.


In [ ]:
from pathlib import Path
import sys

# notebook을 repo 어느 위치에서 열어도 src package를 찾기 위한 bootstrap입니다.
REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "pyproject.toml").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

from labelstudio_bbox_tools.config import settings_from_env

settings = settings_from_env(REPO_ROOT / ".env")
print("repo:", REPO_ROOT)
print("Label Studio URL:", settings.url)
print("doc_root:", settings.doc_root)


In [ ]:
from pathlib import Path
from labelstudio_bbox_tools.importers.annotation_import import import_annotations_to_project

PROJECT_ID = 0
COCO_JSON = Path("/path/to/annotations_mmyolo.json")

# JSON의 image file_name이 어느 root 아래에 있는지 지정합니다.
# fullpath matching을 쓸 때 중요합니다.
MIRROR_ROOT = Path("/path/to/dataset_root")

TARGET_SHAPE = "bbox"
IMAGE_MATCH_MODE = "fullpath"       # "fullpath" 권장, 또는 "basename"
UPLOAD_MODE = "prediction"          # "prediction" 또는 "annotation"

META_TAG = "my_annotation_import"
IMPORT_ID = META_TAG
PRED_MODEL = META_TAG
PRED_TAG = META_TAG

DUMMY_PRED = True
BATCH_SIZE = 200
DRY_RUN = True


In [ ]:
summary = import_annotations_to_project(
    dataset_type="mmyolo",
    ann_source=COCO_JSON,
    project_id=PROJECT_ID,
    ls_url=settings.url,
    api_key=settings.api_key,
    target_shape=TARGET_SHAPE,
    image_match_mode=IMAGE_MATCH_MODE,
    upload_mode=UPLOAD_MODE,
    meta_tag=META_TAG,
    import_id=IMPORT_ID,
    pred_model=PRED_MODEL,
    pred_tag=PRED_TAG,
    dummy_pred=DUMMY_PRED,
    auto_task=False,
    mirror_root=MIRROR_ROOT,
    batch_size=BATCH_SIZE,
    dry_run=DRY_RUN,
)
summary.as_dict()
